# Part 4 — Proving it's right

Part 3 ended by asking how much of what went wrong in parts 2 and 3 dbt's own
testing tools would have caught.

Let's begin by being precise about what is still broken. Parts 2 and 3 fixed
everything that *configuration* can fix: `merge` over `append`, a watermark
on the loader's clock, `hard_deletes: new_record`. Two failures survived that,
and neither is a configuration problem:

1. **the update nobody stamped**: a source that edits a row without touching
  `updated_at`. The snapshot doesn't notice it.
2. **the customer who came back**: a delete-then-resurrect that leaves two open
  versions in the snapshot, even with everything configured correctly.

Both need something to *notice*. This part builds one for the second failure.
The first already got its answer in part 3 (the `check` strategy makes the
change visible, no timestamp required) and stays out of scope here.

**What you'll see**

- the second of those failures turned from silent into loud, in the committed
  project
- the four guards dbt gives you, and the four different moments at which they
  fire
- what `--store-failures` gives you when an error is found at 3am

## Setup

In [1]:
!reset

... restoring raw seed data and dropping dbt schemas


✓ lab restored to its starting state


In [2]:
import pandas as pd

from lab import artifacts, scratch, shop
from lab.runner import dbt, show


def failures() -> list[str]:
    """Names of whatever failed in the most recent invocation."""
    return [uid.split(".")[2] for uid, r in artifacts.run_results().items()
            if r["status"] in ("fail", "error")]


result = dbt("build", quiet=True)
assert result.success

# dbt creates a schema when it materialises a model into it, but the unit-test
# materialisation does not, and this notebook runs a unit test before it builds
# anything. Create the scratch schema up front so that ordering works.
shop.run_sql("IF SCHEMA_ID('scratch') IS NULL EXEC('CREATE SCHEMA scratch')")

m = artifacts.manifest()
print(f"✓ project green · {len(m['nodes']) + len(m['unit_tests'])} nodes")

✓ project green · 41 nodes


---

## The failure that configuration cannot fix

Part 3 left this on the table. Delete a customer, then re-sign them under the
same id, and the snapshot ends up holding two open versions: the deletion
stamped with dbt's clock, the revival with the source's, and no way to order
the two.

Reproduce it:

In [3]:
shop.delete_customer(8)
assert dbt("snapshot", "--threads", "1", quiet=True).success

shop.insert_customer(8, "Hobart Provedores", "TAS", "wholesale", 1200000,
                     updated_at="2026-07-31T09:00:00")
assert dbt("snapshot", "--threads", "1", quiet=True).success

shop.df("""
    SELECT customer_id, segment, dbt_valid_from, dbt_valid_to, dbt_is_deleted
    FROM snapshots.customers_snapshot WHERE customer_id = 8 ORDER BY dbt_valid_from
""")

,customer_id,segment,dbt_valid_from,dbt_valid_to,dbt_is_deleted
0,8,retail,2026-06-02 08:00:00,2026-08-03 12:43:05,False
1,8,wholesale,2026-07-31 09:00:00,NaT,False
2,8,retail,2026-08-03 12:43:05,NaT,True


Two rows with `dbt_valid_to IS NULL`. Any point-in-time join now returns both.

Now the important part. Run the whole project against it, with one exclusion,
so that for a moment the project has exactly the guards part 3 left it with,
and nothing it has gained since:

In [4]:
result = dbt("build", "--exclude", "assert_snapshot_has_one_open_version", quiet=True)
assert result.success
print("✓ green · every guard part 3's project had, and not one of them noticed")

✓ green · every guard part 3's project had, and not one of them noticed


Green. Twenty-nine data tests, an enforced contract, a unit test and a freshness
policy, and not one of them has anything to say about a broken validity chain.

That is not a gap in dbt. It is a gap in what *this project declared*, and the
fix is to declare it. Part 3 promised this test; here it is, committed to
`transform/tests/`, and this time nothing excludes it:

In [5]:
show("tests/assert_snapshot_has_one_open_version.sql")

-- A snapshot must hold exactly one *open* version per key. Nothing in dbt
-- enforces that: a customer deleted and later re-signed under the same id
-- leaves two rows with dbt_valid_to IS NULL, because the deletion is stamped
-- with dbt's clock and the revival with the source's, and the two cannot be
-- ordered. Part 3 finds that failure; this test is what turns it from silent
-- into loud. Returning zero rows is a pass.
select
    customer_id,
    count(*) as open_versions
from {{ ref('customers_snapshot') }}
where dbt_valid_to is null
group by customer_id
having count(*) > 1



In [6]:
result = dbt("build", quiet=True)
assert not result.success
assert "assert_snapshot_has_one_open_version" in failures()
print("✓ the same warehouse, no exclusion · the build fails loudly")

12:43:09  28 of 41 FAIL 1 assert_snapshot_has_one_open_version ........................... [FAIL 1 in 0.07s]


12:43:10  [ERROR]: in test assert_snapshot_has_one_open_version (tests/assert_snapshot_has_one_open_version.sql)


12:43:10    Got 1 result, configured to fail if != 0


✓ the same warehouse, no exclusion · the build fails loudly


Same warehouse, same data: one declaration is the whole difference. The
project gained a guard, and that is the only mechanism part 4 adds.
Everything below is about the four kinds of guard dbt offers and what each
one is blind to.

Put the lab back to a clean state before going on:

In [7]:
shop.reseed_raw()

# Restoring raw is not enough, and part 3 said why: a snapshot is append-only,
# so the versions it already wrote survive any amount of fixing upstream. In a
# lab the way back is to drop it and rebuild from the restored source. In
# production there is no way back, which is the whole reason to get snapshots
# right the first time.
shop.run_sql("DROP TABLE IF EXISTS snapshots.customers_snapshot")

assert dbt("build", "--full-refresh", quiet=True).success
print("✓ restored · the snapshot had to be dropped and rebuilt")


✓ restored · the snapshot had to be dropped and rebuilt


## Four guards, four moments

The guards are easy to confuse because they all "test things". They do not
test the same things, and each fires at a different point: when you edit SQL, when dbt
builds a model, after rows have landed, and whenever you ask about the
loader's clock.

In dbt's own vocabulary only two of these are *tests*: unit tests and data
tests. Contracts and freshness are configuration, but they guard alongside
the tests, so this notebook treats all four as peers.

![four guards, four moments](img/quality-layers.svg)

---

## Unit tests: before any data moves

A unit test checks *transformation logic* against rows you invent. No
warehouse data is read, which means it can run on a laptop, in CI, before
a pull request merges.

The project ships one. It pins the revenue rule in a single sentence:
cancelled orders count as orders, but never as revenue.

In [8]:
show("models/marts/_marts__models.yml")

version: 2

models:
  - name: fct_orders
    description: >
      One row per order, incrementally maintained. The models downstream trust
      this table completely, which is exactly what notebook 02 examines.
    columns:
      - name: order_id
        data_tests: [unique, not_null]
      - name: line_total
        data_tests: [not_null]

  - name: dim_customers
    description: Customers as the source knows them today, with state names.
    columns:
      - name: customer_id
        data_tests: [unique, not_null]

  - name: fct_daily_revenue
    description: >
      Daily order count and revenue (cancelled orders excluded). Carries an
      enforced contract: dbt refuses to build it if the columns drift from
      the shape promised here; the BI layer depends on it.
    config:
      contract:
        enforced: true
    columns:
      - name: order_date
        data_type: date
        constraints:
          - type: not_null
        data_tests: [unique]
      - name: orders_placed
 

Now break the logic it protects. We won't edit the shipped model (a broken
mart would poison every later cell), so this clones it as
`fct_daily_revenue_v2` in a scratch schema, breaks the clone, and points a
copy of the same test at it.

Here is the clone, "simplified" by a developer who assumed only shipped and
delivered orders count. Cancelled orders still earn nothing, which is right.
But `placed` orders now earn nothing either, which is not:

In [9]:
scratch.model("fct_daily_revenue_v2", """
{{ config(materialized='table', schema='scratch') }}
select
    cast(placed_at as date) as order_date,
    count(*) as orders_placed,
    cast(sum(case when status in ('delivered', 'shipped') then line_total else 0 end)
         as decimal(14, 2)) as revenue   -- the "simplification"
from {{ ref('fct_orders') }}
group by cast(placed_at as date)
""")
scratch.yml("_fct_daily_revenue_v2", """
unit_tests:
  - name: v2_placed_orders_still_earn_revenue
    model: fct_daily_revenue_v2
    given:
      - input: ref('fct_orders')
        rows:
          - { order_id: 1, status: placed,    line_total: 100.00, placed_at: 2026-07-01 09:00:00 }
          - { order_id: 2, status: cancelled, line_total: 449.00, placed_at: 2026-07-01 10:00:00 }
    expect:
      rows:
        - { order_date: 2026-07-01, orders_placed: 2, revenue: 100.00 }
""")

result = dbt("test", "--select", "fct_daily_revenue_v2", quiet=True)
assert not result.success
outcome = next(r for uid, r in artifacts.run_results().items() if "v2_placed" in uid)

12:43:13  1 of 1 FAIL 1 fct_daily_revenue_v2::v2_placed_orders_still_earn_revenue ........ [FAIL 1 in 0.13s]


12:43:13  [ERROR]: in unit_test v2_placed_orders_still_earn_revenue (models/scratch/_fct_daily_revenue_v2.yml)


12:43:13    

actual differs from expected:

@@,order_date,orders_placed,revenue
→ ,2026-07-01,2            ,100.00→0.00




In [10]:
assert outcome["status"] == "fail"
print("✓ caught on fixture rows alone · no warehouse data was read")

✓ caught on fixture rows alone · no warehouse data was read


Read that diff: expected revenue 100.00, got 0.00. The fixture is two invented
rows, and it caught a logic regression before the model was ever pointed at
production.

**What it cannot do** is tell you anything about real data. A unit test would
pass happily while every row in your warehouse was wrong.

---

## Contracts: before the model is built

A contract pins a model's **shape**: its columns and their types. With
`contract: { enforced: true }`, dbt compares the SQL's actual output against the
declared shape and refuses to build on a mismatch.

The project puts one on `fct_daily_revenue`, because the BI layer depends on
it. Watch a refactor try to change `revenue` from a decimal to a string:

In [11]:
CONTRACT_YML = """
models:
  - name: api_daily_revenue
    config:
      materialized: table
      schema: scratch
      contract: { enforced: true }
    columns:
      - name: order_date
        data_type: date
      - name: revenue
        data_type: decimal(14,2)
"""
scratch.yml("_api_daily_revenue", CONTRACT_YML)
scratch.model("api_daily_revenue",
              "select order_date, revenue from {{ ref('fct_daily_revenue') }}")
assert dbt("run", "--select", "api_daily_revenue", quiet=True).success
print("✓ the honest version builds")

✓ the honest version builds


In [12]:
scratch.model("api_daily_revenue", """
select order_date, cast(revenue as nvarchar(20)) as revenue
from {{ ref('fct_daily_revenue') }}
""")
result = dbt("run", "--select", "api_daily_revenue", quiet=True)
assert not result.success

# dbt's log above ends in a screen of macro stack; reprint just the part
# that matters: the mismatch table.
message = artifacts.run_results()["model.shop.api_daily_revenue"]["message"]
print(message.split("> in macro")[0].strip())

12:43:14  1 of 1 ERROR creating sql table model scratch.api_daily_revenue ................ [ERROR in 0.05s]


12:43:14  [ERROR]: in model api_daily_revenue (models/scratch/api_daily_revenue.sql)


12:43:14    Compilation Error in model api_daily_revenue (models/scratch/api_daily_revenue.sql)
  This model has an enforced contract that failed.
  Please ensure the name, data_type, and number of columns in your contract match the columns in your model's definition.
  
  | column_name | definition_type | contract_type | mismatch_reason    |
  | ----------- | --------------- | ------------- | ------------------ |
  | revenue     | VARCHAR(8000)   | DECIMAL       | data type mismatch |
  
  
  > in macro assert_columns_equivalent (macros/relations/column/columns_spec_ddl.sql)
  > called by macro default__get_assert_columns_equivalent (macros/relations/column/columns_spec_ddl.sql)
  > called by macro get_assert_columns_equivalent (macros/relations/column/columns_spec_ddl.sql)
  > called by macro sqlserver__create_view_as (macros/relations/views/create.sql)
  > called by macro create_view_as (macros/relations/view/create.sql)
  > called by macro default__get_create_view_as_sql (macros/re

Compilation Error in model api_daily_revenue (models/scratch/api_daily_revenue.sql)
  This model has an enforced contract that failed.
  Please ensure the name, data_type, and number of columns in your contract match the columns in your model's definition.
  
  | column_name | definition_type | contract_type | mismatch_reason    |
  | ----------- | --------------- | ------------- | ------------------ |
  | revenue     | VARCHAR(8000)   | DECIMAL       | data type mismatch |


In [13]:
assert shop.scalar("""
    SELECT DATA_TYPE FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'scratch' AND TABLE_NAME = 'api_daily_revenue'
      AND COLUMN_NAME = 'revenue'
""") == "decimal"
print("✓ build refused · the warehouse still holds the decimal version")

✓ build refused · the warehouse still holds the decimal version


Notice the sequencing. The mismatch was caught *before* any DDL ran, so the
existing table was never touched. Without the contract this ships, and every
downstream `SUM()` starts failing at query time in front of the analysts.

**What it cannot do** is look at a single value. A contract is satisfied by a
table of the right shape containing entirely wrong numbers.

---

## Data tests: after the rows land

These are the ones people mean by "dbt tests". A data test is a query for rows
that should not exist, run against real data after the model is built.

*Generic* tests are declared per column in YAML and reused everywhere. dbt
ships four: `unique`, `not_null`, `accepted_values` and `relationships`
(every value must exist in another model). Between them they cover keys,
enums and referential integrity. This project needs nothing beyond the four:

In [14]:
show("models/staging/shop/_stg_shop__models.yml")

version: 2

models:
  - name: stg_shop__customers
    description: Customers, typed and renamed. Cents became dollars here.
    columns:
      - name: customer_id
        description: Natural key from the operational system.
        data_tests: [unique, not_null]
      - name: segment
        data_tests:
          - accepted_values:
              arguments:
                values: [retail, wholesale, trade]

  - name: stg_shop__products
    description: The catalogue, typed and renamed.
    columns:
      - name: product_id
        data_tests: [unique, not_null]
      - name: sku
        data_tests: [unique, not_null]

  - name: stg_shop__orders
    description: Orders, typed and renamed. `loaded_at` is the loader's clock.
    columns:
      - name: order_id
        data_tests: [unique, not_null]
      - name: status
        data_tests:
          - accepted_values:
              arguments:
                values: [placed, shipped, delivered, cancelled]
      - name: customer_id
       

Break one. Land a second order with an id that already exists, the kind of
thing a loader retry does:

In [15]:
shop.place_order(order_id=7, customer_id=2, product_id=1, quantity=1,
                 unit_price_cents=18900, placed_at="2026-07-30T09:00:00")

result = dbt("build", "--select", "stg_shop__orders", quiet=True)
assert not result.success
print("failed:", failures())

12:43:15  8 of 8 FAIL 1 unique_stg_shop__orders_order_id ................................. [FAIL 1 in 0.06s]


12:43:16  [ERROR]: in test unique_stg_shop__orders_order_id (models/staging/shop/_stg_shop__models.yml)


12:43:16    Got 1 result, configured to fail if != 0


failed: ['unique_stg_shop__orders_order_id']


`unique_stg_shop__orders_order_id` fired, the name dbt builds from the model
and the column. But a failing test only tells you *that* something is wrong.
At 3am you need to know *which rows*. That is what `--store-failures` is for:
it writes the offending rows to a table instead of discarding them.

Note it writes a table for *every* test it runs, not just the failing one, so
most of what appears below is empty.

In [16]:
result = dbt("test", "--select", "stg_shop__orders", "--store-failures", quiet=True)
assert not result.success

stored = [r[0] for r in shop.run_sql(
    "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_SCHEMA = 'dbt_test__audit'")]
print("tables written to dbt_test__audit:")
for name in stored:
    print(" ", name)

failed_rows = next(n for n in stored if n.startswith("unique_"))
shop.df(f"SELECT TOP 5 * FROM dbt_test__audit.[{failed_rows}]")

12:43:17  7 of 7 FAIL 1 unique_stg_shop__orders_order_id ................................. [FAIL 1 in 0.10s]


12:43:17  [ERROR]: in test unique_stg_shop__orders_order_id (models/staging/shop/_stg_shop__models.yml)


12:43:17    Got 1 result, configured to fail if != 0


tables written to dbt_test__audit:
  not_null_stg_shop__orders_customer_id
  accepted_values_stg_shop__orde_d6ffd8874553304dada1919f36a5c44b
  not_null_stg_shop__orders_quantity
  not_null_stg_shop__orders_order_id
  relationships_stg_shop__orders_49852f04eefd50a1a8f6422d5e3d538a
  unique_stg_shop__orders_order_id
  relationships_stg_shop__paymen_87a071c926e65b1dfe91936d2783cf5f


,unique_field,n_records
0,7,2


The duplicated key, sitting in a table you can query.

Now clear the wreckage: restore the raw data, and delete the two deliberately
broken scratch models from earlier; otherwise a full `dbt build` would keep
failing on the very faults this notebook staged on purpose.

In [17]:
shop.reseed_raw()
scratch.clear()
assert dbt("build", "--full-refresh", quiet=True).success
print("✓ raw restored, scratch cleared, project green again")

✓ raw restored, scratch cleared, project green again


### Singular tests: one rule, written in SQL

Generic tests cover expectations that generalise. Some rules do not. This
project ships one written as plain SQL: nothing ships unpaid.

In [18]:
show("tests/assert_payments_cover_shipped_orders.sql")

-- Singular test: a business rule too specific for any generic test.
-- Nothing ships unpaid: every shipped or delivered order must be covered
-- in full by its payments (instalments allowed, hence the SUM).
-- Returning zero rows is a pass; every row returned is a violation.
with orders as (

    select * from {{ ref('fct_orders') }}

),

paid as (

    select
        order_id,
        sum(amount) as amount_paid
    from {{ ref('stg_shop__payments') }}
    group by order_id

),

violations as (

    select
        orders.order_id,
        orders.status,
        orders.line_total,
        coalesce(paid.amount_paid, 0) as amount_paid
    from orders
    left join paid
        on orders.order_id = paid.order_id
    where
        orders.status in ('shipped', 'delivered')
        and coalesce(paid.amount_paid, 0) < orders.line_total

)

select * from violations



In [19]:
shop.update_order_status(16, "shipped", updated_at="2026-07-30T08:00:00")

result = dbt("build", quiet=True)
assert not result.success
assert "assert_payments_cover_shipped_orders" in failures()
print("✓ caught: an order shipped with no payment on file")

# Restoring raw is not enough here, and part 2 explains why: fct_orders already
# merged the shipped row, and the reseeded row's loaded_at is now *below* the
# mart's watermark, so it would never be revisited. Incremental damage needs
# a full refresh.
shop.reseed_raw()
assert dbt("build", "--full-refresh", quiet=True).success
print("✓ restored · with --full-refresh, for the reason part 2 gave")

12:43:22  35 of 41 FAIL 1 assert_payments_cover_shipped_orders ........................... [FAIL 1 in 0.25s]


12:43:22  [ERROR]: in test assert_payments_cover_shipped_orders (tests/assert_payments_cover_shipped_orders.sql)


12:43:22    Got 1 result, configured to fail if != 0


✓ caught: an order shipped with no payment on file


✓ restored · with --full-refresh, for the reason part 2 gave


**What data tests cannot do** is check anything nobody declared. There is no
test here for "quantities are positive" or "revenue is roughly yesterday's
revenue", because nobody wrote one. Part 5 measures exactly how much that
matters.

---

## Source freshness: the loader's clock

The last guard watches neither shape nor values, but *time*. Source
declarations can carry a freshness policy, and `dbt source freshness` checks
the newest `_loaded_at` against it.

This lab's landing zone was seeded with data frozen in July 2026, so by the
time you run this it is long stale, which makes failure the correct result,
permanently:

In [20]:
result = dbt("source", "freshness", quiet=True)
freshness = artifacts.freshness()

assert result.success is False
assert "error" in freshness.values()
pd.DataFrame([{"source": source, "status": status} for source, status in freshness.items()])

12:43:26  1 of 1 ERROR STALE freshness of shop.orders .................................... [ERROR STALE in 0.03s]


12:43:26  [ERROR]: in source orders (models/staging/shop/_shop__sources.yml)


12:43:26    Status: error


,source,status
0,orders,error


A pipeline that runs perfectly on raw data that has not refreshed for days will pass
every other check in this notebook. This is the only guard that would notice.

---

## What this notebook showed

| guard | checks | fires | blind to |
|---|---|---|---|
| unit tests | transformation logic | before merge, on fixtures | real data entirely |
| contracts | a model's columns and types | before the model builds | values entirely |
| data tests (generic) | declared per-column facts | after rows land | anything undeclared |
| data tests (singular) | one business rule each | after rows land | every rule nobody wrote |
| source freshness | the loader's clock | on demand | the content of what loaded |

The project now carries 30 data tests (one more than part 3 left it with), an
enforced contract, a unit test and a freshness policy. Be precise about what
that demonstrated: three of those tests fired here, on faults staged for them,
and the contract and unit-test demos ran against throwaway clones so the
shipped project would stay green. The other 27 data tests were never challenged.

That is the normal state of a well-tested project (most guards are silent
most of the time), and it invites a comfortable conclusion: the pipeline is
covered. That conclusion is worth measuring rather than assuming.

*Next, fifteen realistic defects thrown at our configured project: [the audit →](05-the-audit.ipynb)*